# M5P2 IGV on GSE128157

In this notebook, we will use IGV on all of our samples within our STAR alignment experiment. We will also use STAR again on the same samples to quantify transcripts and perform some processing with stringtie for novel isoform quantification as a proof of concept.
Objectives: 
- Generate indexes for each sample's `BAM` file
- Load each file into IGV for use in this week's exercise
- Run a `STAR` alignment again, specifically for the use of `stringtie`
- Use `stringtie` to identify potential novel transcripts within the samples, and investigate them


```{note}
**Runs on an HPC.** This notebook was written for the Hellbender HPC cluster (University of Missouri) and uses SLURM and/or large sequencing files. It is shown here for reading; saved outputs were removed for privacy. Set `REPO_DIR`, `DATA_DIR` and `WORK_DIR` (see the repository README) and download inputs with `data/fetch_data.sh` before running it on your own cluster.
```

In [ ]:
export TERM=xterm

ml mamba

source activate module_5_IGV

cd ../m5_project_STAR/

We will now create an index for each of the samples within our folder `m5_project_STAR/results/alignments`. This will be used by IGV to open our samples. 

In [ ]:
set -euo pipefail

ALIGN_DIR="results/alignments"

BAM_FILES=("$ALIGN_DIR"/*.bam)

# samtools index loop
for bam in "${BAM_FILES[@]}"; do
    echo "Indexing $bam"
    samtools index "$bam"
done

echo "All BAM files indexed."


Now, you should watch this video on making comparisons between our samples within IGV. We will perform a similar surface-level analysis on a DEG of your choice in the exercise this week. 

## [M5P2V5 - Use of IGV on GSE128157](https://youtu.be/RTwHYyz58Mg)

The remainder of this practice will be dedicated to using `StringTie`, a program that can detect novel isoforms. This is superficial, as it is very unlikely the OPT3 experiment will result in novel isoforms, but it is an important skill nonetheless.

# StringTie: Transcript Assembly and Quantification

**StringTie** is a computational tool used in RNA-Seq analysis to assemble and quantify transcripts from aligned RNA-Seq reads. It is particularly useful for detecting novel isoforms and estimating transcript abundance.

 StringTie takes genome-aligned BAM files and performs two main tasks:

1. **Transcript Assembly**  
   - Reconstructs full-length transcripts from overlapping RNA-Seq reads.
   - Can identify both **known** transcripts (from an annotation) and novel isoforms.
   - Uses splice junction information to correctly assemble exons.

2. **Transcript Quantification**  
   - Estimates the abundance of each transcript in terms of FPKM or TPM.
   - Can perform gene-level or transcript-level quantification.
   - Can merge multiple samples into a single unified transcriptome using `stringtie --merge`.
  
`STAR` must be run with these parameters, as recommended from the [StringTie Documentation](https://github.com/gpertea/stringtie), for the tool to work: 

    --outSAMstrandField intronMotif
    --alignIntronMin 20
    --alignIntronMax 1000000
    --alignMatesGapMax 1000000

Run `STAR` again, using these parameters:


In [ ]:
set -euo pipefail

# Directories & threads
TRIMMED_DIR="data/trimmed"
ALIGN_DIR="results/stringtie_alignments"
STAR_INDEX="references/star_index"
THREADS=8

# Temporary directory for STAR
TMP_DIR="/tmp/STAR_tmp"
mkdir -p "$ALIGN_DIR" "$TMP_DIR"

shopt -s nullglob
FASTQ_FILES=("$TRIMMED_DIR"/*_R1_trimmed.fastq.gz)
if [ ${#FASTQ_FILES[@]} -eq 0 ]; then
    echo "No R1 FASTQ files found in $TRIMMED_DIR. Exiting."
    exit 1
fi

# STAR loop
for fq1 in "${FASTQ_FILES[@]}"; do
    sample=$(basename "$fq1" _R1_trimmed.fastq.gz)

    echo "Running STAR for sample $sample"

    STAR --runThreadN "$THREADS" \
         --genomeDir "$STAR_INDEX" \
         --readFilesIn "$fq1" \
         --readFilesCommand zcat \
         --outFileNamePrefix "$ALIGN_DIR/${sample}_" \
         --outSAMtype BAM SortedByCoordinate \
         --outTmpDir "$TMP_DIR/${sample}" \
         --outSAMstrandField intronMotif \
         --alignIntronMin 20 \
         --alignIntronMax 1000000 \
         --alignMatesGapMax 1000000

    echo "Finished STAR for sample $sample"
done

echo "All samples processed successfully!"


# StringTie Parameters Reference

| Parameter | Description | Typical Use / Notes |
|-----------|-------------|-------------------|
| `-G` | Reference annotation GTF file | Guides transcript assembly; allows detection of known and novel isoforms. |
| `-o` | Output GTF file | File where assembled transcripts and abundance estimates are written. |
| `-p` | Number of threads | Speeds up processing; usually set to available cores. |
| `-e` | Estimate expression only for input annotation | Used with a merged GTF to quantify known transcripts without assembling new ones. |
| `-B` | Prepare Ballgown input | Generates files for downstream differential expression analysis with Ballgown in R. |
| `--merge` | Merge multiple GTFs | Creates a unified transcriptome from multiple sample assemblies. |
| `-f` | Minimum isoform fraction | Filters out low-abundance isoforms (default 0.1). |
| `-c` | Minimum read coverage | Filters transcripts with low read coverage (default 1). |
| `-M` | Maximum fraction of multi-hit reads allowed in a transcript | Helps filter unreliable transcript predictions. |
| `-l` | Prefix for new transcript IDs | Useful when merging multiple samples to avoid ID collisions. |

**Notes:**

- Always use genome-aligned BAM files as input, not transcriptome alignments.  
- Common workflow: per-sample assembly > merge GTF > quantification with `-e -B`.  
- Adjust `-f` and `-c` to control stringency of novel isoform detection.

Now, we will run `StringTie` on these samples:


In [ ]:
set -euo pipefail

# Directories
ALIGN_DIR="results/stringtie_alignments"
STRINGTIE_DIR="results/stringtie/per_sample/"
GTF_REF="references/annotation/Arabidopsis_thaliana.TAIR10.62.gtf" 
THREADS=8

mkdir -p "$STRINGTIE_DIR"

# Loop over STAR BAMs
BAM_FILES=("$ALIGN_DIR"/*_Aligned.sortedByCoord.out.bam)
if [ ${#BAM_FILES[@]} -eq 0 ]; then
    echo "No BAM files found in $ALIGN_DIR. Exiting."
    exit 1
fi
#run stringie
for bam in "${BAM_FILES[@]}"; do
    sample=$(basename "$bam" _Aligned.sortedByCoord.out.bam)
    
    echo "Running StringTie for sample $sample"

    stringtie "$bam" \
              -p "$THREADS" \
              -G "$GTF_REF" \
              -o "$STRINGTIE_DIR/${sample}.gtf" \
              -A "$STRINGTIE_DIR/${sample}_gene_abund.tab" \
              -e -B

    echo "Finished StringTie for sample $sample"
done

echo "All samples processed with StringTie successfully!"


We can now merge all of the samples into a single transcriptome to see if there are any novel isoforms when we compare it to our original TAIR10 `gtf`.

In [ ]:
ls results/stringtie/per_sample/*.gtf > results/stringtie/sample_assembly_list.txt

stringtie --merge -G references/annotation/Arabidopsis_thaliana.TAIR10.62.gtf -o results/stringtie/merged_samples.gtf results/stringtie/sample_assembly_list.txt

Now, we use `gffcompare` to see if any of the transcripts that are found within our `merged_samples.gtf` do not exist witihn the TAIR10 annotation, and parse the outputs for any potenital novel isoforms:

In [ ]:
gffcompare -r references/annotation/Arabidopsis_thaliana.TAIR10.62.gtf -G -o results/stringtie/gffcmp results/stringtie/merged_samples.gtf
# main outputs: results/stringtie/gffcmp.stats and gffcmp.annotated.gtf and gffcmp.tracking

In [ ]:
# class codes are in attribute field like 'class_code "j";'
grep 'class_code "j"' results/stringtie/gffcmp.annotated.gtf > novel_isoforms_j.gtf
grep 'class_code "u"' results/stringtie/gffcmp.annotated.gtf > novel_loci_u.gtf

# list transcript IDs
awk -F'\t' '/class_code "j"/{match($9, /transcript_id "([^"]+)"/,a); print a[1]}' results/stringtie/gffcmp.annotated.gtf | sort | uniq > novel_isoform_ids.txt


Sadly, in our case, the *A. thalina* genome is too well annotated, but the pipeline holds up. Check out the `gffcmp.stats` file within `m5_project_STAR/results/stringtie/` for a summary of the `stringtie` results. 

Spend the rest of the week working on your write-up, as well as making sure to look for a data set for use in the final project. If you cannot find a data set, feel free to reach out to the Module 5 Help and Questions Discussion on Canvas!